In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model    import Ridge
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics         import mean_absolute_error, r2_score

# ── 1. Load cleaned data ──────────────────────────────────────
df = pd.read_csv("C:\\Users\\User\\Desktop\\developer survey ML project\\data\\survey_cleaned.csv")
print("Loaded:", df.shape)

Loaded: (4149, 38)


In [4]:
# ── 2. Define X and y ─────────────────────────────────────────
X = df.drop(columns=["salary", "log_salary"])
y = df["log_salary"]

print("Features:", X.shape[1])
print("Target mean (log):", y.mean().round(3), "→ exp =", round(np.exp(y.mean()), 0))

Features: 36
Target mean (log): 11.891 → exp = 145946.0


In [5]:
# ── 3. Train/test split ───────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("\nTrain rows:", len(X_train))
print("Test rows: ", len(X_test))


Train rows: 3319
Test rows:  830


In [6]:
# ── 4. Scale features ─────────────────────────────────────────
# fit_transform on train only — then only transform on test
# if you fit on test data too, you're leaking test info into the model
scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),      columns=X.columns)

# Quick sanity check — after scaling, mean≈0 and std≈1
print("\nAfter scaling:")
print("  work_exp mean:", X_train_sc["work_exp"].mean().round(3))
print("  work_exp std: ", X_train_sc["work_exp"].std().round(3))



After scaling:
  work_exp mean: -0.0
  work_exp std:  1.0


In [7]:
# ── 5. Train Ridge regression ─────────────────────────────────
model = Ridge(alpha=10)
model.fit(X_train_sc, y_train)
print("\nModel trained.")


Model trained.


In [8]:
# ── 6. Evaluate ───────────────────────────────────────────────
y_pred_log    = model.predict(X_test_sc)
y_pred_dollars = np.exp(y_pred_log)
y_test_dollars = np.exp(y_test)

mae  = mean_absolute_error(y_test_dollars, y_pred_dollars)
r2   = r2_score(y_test, y_pred_log)
mape = np.mean(np.abs((y_test_dollars - y_pred_dollars) / y_test_dollars)) * 100

print("\n── Test Results ──────────────────────────")
print(f"  MAE  : ${mae:,.0f}   ← avg dollar error")
print(f"  MAPE : {mape:.1f}%      ← avg % error")
print(f"  R²   : {r2:.3f}      ← variance explained")


── Test Results ──────────────────────────
  MAE  : $45,107   ← avg dollar error
  MAPE : 33.9%      ← avg % error
  R²   : 0.283      ← variance explained


In [9]:
# ── 7. Cross-validation ───────────────────────────────────────
cv_scores = cross_val_score(Ridge(alpha=10), X_train_sc, y_train, cv=5, scoring="r2")
print(f"\n  5-fold CV R²: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"  Folds: {cv_scores.round(3)}")



  5-fold CV R²: 0.334 ± 0.055
  Folds: [0.262 0.278 0.356 0.41  0.362]


In [10]:
# ── 8. Coefficient table ──────────────────────────────────────
coef = pd.Series(model.coef_, index=X.columns)
coef_sorted = coef.reindex(coef.abs().sort_values(ascending=False).index)

print("\n── Top 15 features by coefficient ────────")
print(f"  {'Feature':<25} {'Coef':>8}   Effect")
print(f"  {'-'*45}")
for feat, val in coef_sorted.head(15).items():
    direction = "↑ higher salary" if val > 0 else "↓ lower salary"
    print(f"  {feat:<25} {val:>+8.4f}   {direction}")


── Top 15 features by coefficient ────────
  Feature                       Coef   Effect
  ---------------------------------------------
  log_work_exp               +0.2992   ↑ higher salary
  org_size                   +0.1320   ↑ higher salary
  exp_x_edu                  -0.0959   ↓ lower salary
  work_exp_sq                -0.0945   ↓ lower salary
  education                  +0.0806   ↑ higher salary
  work_exp                   +0.0408   ↑ higher salary
  cloud_aws                  +0.0393   ↑ higher salary
  is_fullstack               -0.0390   ↓ lower salary
  remote_work                +0.0329   ↑ higher salary
  exp_x_orgsize              -0.0323   ↓ lower salary
  years_code                 +0.0301   ↑ higher salary
  is_backend                 +0.0300   ↑ higher salary
  lang_go                    +0.0276   ↑ higher salary
  cloud_azure                -0.0266   ↓ lower salary
  n_hv_langs                 +0.0239   ↑ higher salary


In [11]:
import pickle
import os

os.makedirs("models", exist_ok=True)

# Save model
with open("models/model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save scaler — must save this too
# If you use a different scaler object later, predictions will be wrong
with open("models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Save feature names — so we never mix up column order
with open("models/feature_names.pkl", "wb") as f:
    pickle.dump(list(X.columns), f)

# Save test data — so SHAP notebook uses exact same test split
X_test_sc.to_csv("models/X_test_sc.csv", index=False)
y_test.to_csv("models/y_test.csv", index=False)

print("Saved:")
print("  models/model.pkl")
print("  models/scaler.pkl")
print("  models/feature_names.pkl")
print("  models/X_test_sc.csv")
print("  models/y_test.csv")

Saved:
  models/model.pkl
  models/scaler.pkl
  models/feature_names.pkl
  models/X_test_sc.csv
  models/y_test.csv
